In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from patsy import dmatrix
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.nonparametric.smoothers_lowess import lowess

# Building a model
In this section we will look at pratical example of how we can build a regression model to see how whether membrane resistance, spike threshold and capacitance contribute to rheobase. Often we just want to test whether our measure is different between genotype, treatment, sex, etc. However, we can actually build a more biological plausible model that also gets into some simple causal analysis. This analysis will also cover several techniques such as model comparison, residual analysis and intepretation of regression coefficients.

## Setting the baseline
First we are going to see how well predicted rheobase matches up to the actual rheobase. We can use the simple equation $\Delta V=IR_m$ for the relationship between resistance and rheobase. To get $\Delta V$  we will subtract the resting membrane potential from the spike threshold. To get $I$ we will divide $\Delta V$ by $R_m$. We will regress the predicted rheobase against the actual rheobase using a simple linear regression. Then we will analyze the regression output.

### Load and prepare the data
The dataset being used for this chapter is the same data used for the MSN data in other chapters. I and others recorded D1-tdTom+ and D1-tdTom- cells from adult mice (~P90). A couple notes on how the data was analyzed. Membrane resistance was measured using the currents injections from -150 to 150 and is the slope coefficient of the regression `current ~ delta_v`. A slow current was injected to keep the cells close to -70 mV. The spike threshold was measured using the peak of the third derivative. Capacitance (Cm) was measured on break-in using a test pulse. To prepare our dataset we will need to rename our columns so we can use the statsmodels formula API. This means we need to remove spaces, parenthesis, brackets, etc. for the column names.

In [ ]:
data_path = Path().cwd().parent / "data/stats/msn_data.csv"
df = pd.read_csv(data_path)

df = df.dropna(subset="Vm_B", axis="rows")
df["deltav_mem"] = df["Spike threshold (mV)"] - df["Vm_B"]
df["synth_rheo"] = (df["deltav_mem"] / df["Membrane resistance"]) * 1000
df["mem_res"] = df["Membrane resistance"]
df["inv_mem_res"] = 1/df["mem_res"]
df["rheo"] = df["Rheobase (pA)"]
df["log_rheo"] = np.log(df["Rheobase (pA)"])
df["log_mem"] = np.log(df["mem_res"])
df["spk_thresh"] = df["Spike threshold (mV)"]
df["log_spk_thresh"] = np.log((df["spk_thresh"] - df["spk_thresh"].min())+1e-10)
df["sqrt_spk_thresh"] = np.sqrt((df["spk_thresh"] - df["spk_thresh"].min())+1e-10)

### Run the regression

In [ ]:
model_fit = smf.ols("rheo ~ synth_rheo", data=df).fit()

Plot the resulting fit and the points.

In [ ]:
fig, ax = plt.subplots()
ax.plot(df["synth_rheo"], df["Rheobase (pA)"], ".")
x = np.linspace(df["synth_rheo"].min(), df["synth_rheo"].max(), num=100)
fit = model_fit.params["Intercept"] + model_fit.params["synth_rheo"]*x
ax.plot(x, fit)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

The fit looks goods, but how good is it? Notice how the predicted rheobase seems about 2x larger than the actual rheobase. We can look at the actual model to get more information about this relationship.
### Analyze the regression

In [ ]:
print(model_fit.summary())

Some things to note about the model summary. Our slope coefficent, `synth_rheo` is 0.3509 which means if we compare one cell with a rheobase of say 100 and one with a rheobase of 101 the difference in actual rheobase between the two cells would be 0.3509. This means our input needs to be multiplied by 0.3509 to get to the actual rheobase. This is pretty close to the estimate of the predicted rheobase being 2x larger than the actual rheobase. Why is the slope not close to one? It might depend on how we measure membrane resistance. For this data set the cell was held at -70 mV by injecting a slow (DC) current. I regressed the delta v against the current injection amplitude only for acquisition with 150 pA of the 0 pA current injection. We have different sets of ion channels that are active at current injections. *Ih* channels open below -40 mV, Na+ channels start to open with depolarizing current injections, K+ leak channels are probably more important around the resting membrane potential just to name a few. What if we break down the predicted rheobase into its components? Lastly we should look at the confidence intervals to get an idea of how good our estimate is. Even though `p < 0.05` this does not mean our estimate is precise it just means the confidence intrevals do not contain zero. A rule of thumb is if you are running and OLS or some variant such as weighted or robust regression, you do not log transform your outcome variable and your confidence intervals do not contain zero the take the ratio of the intervals where the absolute value of the larger interval is on top: `abs(large)/abs(small)`. A rule of thumb is if this interval is > 3 then we have a weak effect. If you have log transformed your outcome then you need to exponentiate your coefficient and confidence intervals first. The exponentiated confiden. So for regression `0.408/0.294=1.388`. So we can say that we are fairly confident in our slope measurement regardless of the p-value. If your confidence intervals contain zero then you need to decide on something called a region of pratical equivalence using two one-sided tests (TOST). Or you can decide on an effect size that is practical. Lastly we can look at `Adj. R-squared` to see how much variance is explained by our model. `Adj. R-squared` is like `R-squared` except that it is *adjusted* for sample size and number of coefficients. `Adj. R-squared` and `R-squared` actually converge when the estimated relationship is strong or you have a large sample size. One problem with `Adj. R-squared` is it is supposed to show us how variance is explained by our model but, it can be negative. This does not really matter because a `R-squared` close to zero means we explain vary little about the outcome variable with our model. `Adj. R-squared` at 0.654 is pretty good. That means 64% of our outcome is explained by our predictors. Later we will compare the `Adj. R-squared` of different models to get a sense of how big 0.654 really is.
 
## Begining to breakdown the model
First let's look at how the different components may be related.

In [ ]:
columns = [
    "rheo",
    "mem_res",
    "spk_thresh",
    "Vm_B",
    "Cm",
    "Rs_B",
]
fig, ax = plt.subplots(
    nrows=len(columns), ncols=len(columns), layout="constrained", figsize=(15, 15)
)
values = len(columns)
for i in range(values):
    for j in range(values):
        if i != j:
            x = df[columns[i]]
            y = df[columns[j]]
            model_fit = smf.ols(f"{columns[j]} ~ {columns[i]}", data=df).fit()
            x_fit = np.linspace(df[columns[i]].min(), df[columns[i]].max(), num=100)
            y_fit = model_fit.params["Intercept"] + model_fit.params[columns[i]]*x_fit
            ax[i][j].plot(x, y, ".")
            ax[i][j].set_xlabel(columns[i])
            ax[i][j].set_ylabel(columns[j])
            ax[i][j].plot(x_fit,y_fit)
        else:
            ax[i][j].hist(df[columns[i]], bins=15)
            ax[i][j].set_xlabel(columns[j])

Probably the two biggest things that stand out are that membrane resistance and spike threshold seem to have some relationship with rheobase. The relationship between rheobase and membrane resistance is nonlinear. This makes sense since rheobase and membrane resistance are inversely proportional. Spike threshold is also fairly linearly correlated with spike threshold. While there seem to be some linear relations between other variables the variance around the simple regression line is quite a lot. When we build a regression model with more predictors we want to avoid collinear features since this will impair the regression fit. Also if we look at the histogram distributions we can see that most variables are left skewed (tail to the right).  It is also interesting that membrane resting potential does not seem to be related the membrane resistance. This could be because of how we measure the membrane resistance. Since I injected a slow current to keep the cells as -70 mV this could change the types of ion channels that are being sampled.

So this means a couple things. We will likely need a transform to run regression with coefficients for each component. To breakdown $\Delta V=IR$ we are not going to run a model with the formula `rheobase~inv_mem_res+spk_thresh+Cm`. One is that there is a multiplicative relationship between `inv_mem_res` and `spk_thresh` so treating it as additive will not work. We can see that most of the variables are skewed to the right which means we can log transform them with the exception of and parameters that are negative. These values would have to be shifted and log transformed or left alone.

## Log transforming the variables

In [ ]:
fig, ax = plt.subplots(ncols=3, figsize=(10,4), layout="constrained")
for index, i in enumerate(["log_rheo", "log_mem", "sqrt_spk_thresh"]):
    ax[index].hist(df[i], bins=15)

I ended up using a log transform for both membrane resistance and rheobase. They look fairly normally distributed. It is important to note that your variables do not have be normally distributed to run an OLS however, if they are right skewed then most of the time it will improve the model fit. Another thing to notice is that there seem to be a couple of potential outliers in the transformed spike threshold at the lower end where are for the untransformed data they seemed to be higher values. At this point it is hard to say whether we should use a transformed or untransformed. We could compare two regression models and see which is better but in this case we will stick with the untransformed unless something in our model indicates that we need to transform it. We could also look back and see if the dataset was cleaned appropiately. We did not transform capacitance (Cm). The real potential problem with capacitance is that it is bimodal. As far as I can tell there is not data that I collected that can split these groups into two such as whether the cells were part of the patch or matrix. 

## Comparing the transformed data.
We can plot the newly transformed data to see how everything relates to rheobase with just a simple regression.

In [ ]:
plot_pairs = [("log_mem", "log_rheo"), ("Cm", "log_rheo"), ("spk_thresh", "log_rheo"), ("Vm_B", "log_rheo")]
fig, ax = plt.subplots(ncols=len(plot_pairs), layout="constrained", figsize=(10,3))
for index, i in enumerate(plot_pairs):
    ax[index].plot(df[i[0]], df[i[1]], ".")
    ax[index].set_title(f"{i[0]}, {i[1]}")
    model_fit = smf.ols(f"{i[1]} ~{i[0]}", data=df).fit()
    x_fit = np.linspace(df[i[0]].min(), df[i[0]].max(), num=100)
    y_fit = model_fit.params["Intercept"] + model_fit.params[i[0]] * x_fit
    ci = tuple(np.exp(model_fit.conf_int().loc[i[0]]))
    param = np.exp(model_fit.params[i[0]])
    ax[index].plot(x_fit, y_fit)

Membrane resistance and spike threshold look to have a very strong relationship with rheobase while capacitance and Vm_B not so much. One last thing we should check before building a model is collinearity. Collinearity prevents an OLS regression model from finding a single, unique solution. We can either plot the relationship between the variables or we can run something called variance inflation factor (VIF) analysis. We will use this so you know how to use it. We can also check the regression covariance parameters which we will do later.

## VIF

In [ ]:
exog = dmatrix("~ log_mem + spk_thresh + Cm + Vm_B", data=df)
vifs = [variance_inflation_factor(exog, i ) for i in range(exog.shape[-1])]
for i, j in zip(exog.design_info.column_names, vifs):
    print(f"{i}: {j}")

As a rule of thumb we are looking for VIFs greater that five. You might notice that the intercept is super high but, we can ignore the intercept. In fact, if we center all of our continuous variables, the intercept will just be one. All of the values are below five so we are good to go build a more complicated regression model for rheobase.

## Building and comparing regression models.
There are two ways you can build models. Start simple and add factors. Start will all your factors and remove those that are not improving the fit. You really want to have a predefined model though and then use model comparison to explain why certain variables can be remove from the model. You can also use regularization for models with many predictors (think like 20+). Ideally, if you want to test genotype and sex you should keep in both variables or use model comparison to explain why you are using the simple model instead.

### AIC for model comparison
One of the best ways to compare different models is to use Akake Information Criterion (AIC). AIC is based on the negative log maximum likeihood fit (lower number is better) and penalizes for the number of parameters in the model. Generally if a model has a lower AIC by about 10 points or more compared to another model, than the model with lower AIC is much better. A 5 point different is not quite as good but is still good enough to choose the lower model. We can also run an F-test to compare models and get a p-value. Other regression fit indicators like R-squared will keep increasing as you add predictor variables. Lastly, if you are transforming your outcome you cannot compare a model with a transformed outcome to one without because the scale of the data changes so Adj. R-squared and AIC will automatically be different and usually lower even if the model does not fit well.

## Interpreting coefficients for log transformed data
Interpreting the coefficents for log transformed data is more complicated. We are still looking to see how far the coefficent is from 0. But it is a multiplicative scale. P-values will be significant even if the 

### Comparing simple models
We will compare some simple regression models. I would usually not do it this way but it is informative to see how AIC and Adj. R-squared are.

In [ ]:
for index, i in enumerate([
    "log_rheo ~ log_mem",
    "log_rheo ~ spk_thresh",
    "log_rheo ~ Cm",
    "log_rheo ~ Vm_B",
    "log_rheo ~ C(D1_D2, Sum)",
]):
    model = smf.ols(i, df).fit()
    ci = [round(i, 3) for i in model.conf_int().iloc[1]]
    print(
        f"Model {index}, AIC: {model.aic:.3f}, Adj. R: {model.rsquared_adj:.4f}, Coeff: {model.params.iloc[1]:.3f}, CI: {ci}"
    )

We see that spike threshold and membrane resistance contribute can explain quite a bit of variable in the model. We see that Vm_B contributes almost nothing which is interesting since I have seen papers using membrane voltage alone to claim changes in excitability. Additionally, we can see that Cm and Vm_B alone do not significantly contribute to rheobase. Now one important factor when we move to a more complex model each of these variables effects may change when other variables are added to the model. Since membrane resistance contributes the most to the model we will start with that are add terms to the model to see if they improve the fit. Another thing we have not done is assessed the quality of the fit by looking at the residuals.

### Comparing more complex models.

In [ ]:
models = []
for index, i in enumerate([
    "log_rheo ~ log_mem",
    "log_rheo ~ log_mem + spk_thresh",
    "log_rheo ~ log_mem + spk_thresh + Cm",
    "log_rheo ~ log_mem + spk_thresh + Vm_B",
    "log_rheo ~ log_mem + spk_thresh + Cm + Vm_B",
]):
    model = smf.ols(i, df).fit()
    models.append(model)
    ci = [round(i, 3) for i in model.conf_int().iloc[1]]
    print(
        f"Model {index}, AIC: {model.aic:.3f}, Adj. R: {model.rsquared_adj:.4f}"
    )

Based on the AIC we can see that adding spike threshold to memebrane resistance greatly improves the model fit. We see that capacitance (Cm) improves is a little bit, probably enough to just keeping it in. However, resting membrane potential does not seem to add much. So our full model is `log_rheo ~ log_mem + spk_thresh + Cm`. Lets see how it actually fits by looking at the residuals.

In [ ]:
fig, ax = plt.subplots(ncols=3, figsize=(10,3), layout="constrained")
index = 2
influence = models[index].get_influence()
resids = influence.resid_studentized_internal
ax[0].hist(resids, bins=15)
ax[1].axhline(0, linestyle=":", color="black", linewidth=3)
ax[1].plot(models[index].fittedvalues, resids, ".", color="magenta", alpha=0.6, markersize=10, markeredgecolor="none")
w = lowess(resids, models[index].fittedvalues)
ax[1].plot(w[:,0], w[:, 1], color="orange", linewidth=2)
ax[1].set_xlabel("Fitted values")
ax[1].set_ylabel("Studentized Residuals")
ax[1].set_title("Residuals vs Fitted")
ax[2].set_title("Scale-location")
ax[2].plot(models[index].fittedvalues, np.sqrt(np.abs(resids)), ".", color="magenta", alpha=0.6, markersize=10, markeredgecolor="none")
w = lowess(np.sqrt(np.abs(resids)), models[index].fittedvalues)
ax[2].plot(w[:,0], w[:, 1], color="orange", linewidth=2)

First we see that the distribution of residuals is fairly normal. QQ plots are often used however they are not actually that useful. The residual histogram is just anther way to see how the residuals are distributed. The residuals vs the fitted plot show us that we have a good linear relationship as shown by the mostly flat orange line. A little wiggle is okay. In the scale-location plot, the orange line trends upwards.This indicates increasing variance as rheobase increases. This is interesting biologically too. Why would rheobase get more variable as the cell gets less excitable? This could be measurement error compounded with biophysical bounds. The heteroscedasticity violation is much more important than the normality of the residuals. We will try two more things fix to the model before analyzing the coefficients.

### Assess the coefficients

In [ ]:
print(models[2].summary())

In [ ]:
np.exp([-0.005, 0])

First thing I notice is that we have a large effect of membrane resistance and smaller effect of spike threshold. While there is a capcitance effect we see that it is tiny

### Weighted least squares
One common and easy way to correct for heteroscedasticity is to run a weighted least squares (WLS). We add a weights argument to WLS to downweight high variance points. To do this we need to through a couple steps. 
1. Square the resids from the first model to ensure there is no negative variance. 
2. After squaring we log transform the residuals. 
3. Then we regress the residuals as the outcome and the fitted values as the predictor. 
4. Then we exponentiate the fitted_variance from the new regression to get the residuals on the scale we need.
5. We run a WLS model with weights as `1/fitted_variance`.
6. To assess the residuals of the WLS get the raw residual and divide by the sqrt of the fitted variance.

In [ ]:
variance_model = smf.ols(
    "resid ~ fittedvalues",
    {"resid": np.log(resids**2 + 1e-6), "fittedvalues": models[2].fittedvalues},
).fit()
fitted_variance = np.exp(variance_model.fittedvalues)
wls_model = smf.wls(
    "log_rheo ~ log_mem + spk_thresh + Cm", data=df, weights=1 / fitted_variance
).fit()
fig, ax = plt.subplots(ncols=3, figsize=(10, 3), layout="constrained")
wls_resids = wls_model.resid / np.sqrt(fitted_variance)
ax[0].hist(wls_resids, bins=15)
ax[1].axhline(0, linestyle=":", color="black", linewidth=3)
ax[1].plot(
    wls_model.fittedvalues,
    wls_resids,
    ".",
    color="magenta",
    alpha=0.6,
    markersize=10,
    markeredgecolor="none",
)
w = lowess(wls_resids, wls_model.fittedvalues)
ax[1].plot(w[:, 0], w[:, 1], color="orange", linewidth=2)
ax[1].set_xlabel("Fitted values")
ax[1].set_ylabel("Studentized Residuals")
ax[1].set_title("Residuals vs Fitted")
ax[2].set_title("Scale-location")
ax[2].plot(
    wls_model.fittedvalues,
    np.sqrt(np.abs(wls_resids)),
    ".",
    color="magenta",
    alpha=0.6,
    markersize=10,
    markeredgecolor="none",
)
w = lowess(np.sqrt(np.abs(wls_resids)), wls_model.fittedvalues)
ax[2].plot(w[:, 0], w[:, 1], color="orange", linewidth=2)

The fit is good what about the summary?

In [ ]:
print(wls_model.summary())

So if we look at the coefficients we can see that membrane resistance has the largest effect, followed by spike threshold and finally capacitance.
Even though capacitance has a p-value that is significant the effect is tiny. So capacitance is "soaking" up some of the variance.